In [0]:
from pyspark.sql.functions import current_date, lit

# Simulated incoming source data (Incremental)
source_df = spark.createDataFrame([
    ("C001", "Niranjan", "Bangalore"),
    ("C001", "Niranjan", "Bankok"),  # City changed from Mumbai → Bangalore
    ("C002", "Rahul", "Pune"),          # No change
    ("C004", "Asha", "Chennai")         # New Customer
], ["customer_id", "customer_name", "city"])

source_df = source_df.withColumn("start_date", lit(None)).withColumn("end_date", lit(None)).withColumn("is_current", lit("Y"))


In [0]:
source_df.write.mode("overwrite").format("delta").saveAsTable("Sales123")

In [0]:
%sql
select * from DimCustomer;

In [0]:
from pyspark.sql.functions import current_date, lit

# Simulated incoming source data (Incremental)
source_df = spark.createDataFrame([
    ("C001", "Niranjan", "Bangalore"),  # City changed from Mumbai → Bangalore
    ("C002", "Rahul", "Pune"),          # No change
    ("C004", "Asha", "Chennai")         # New Customer
], ["customer_id", "customer_name", "city"])

source_df = source_df.withColumn("start_date", lit(None)).withColumn("end_date", lit(None)).withColumn("is_current", lit("Y"))


In [0]:
dim_df = spark.table("DimCustomer")
dim_df.display()


In [0]:
%sql
select * from DimCustomer;

In [0]:
from pyspark.sql.functions import sha2, concat_ws

# Hash to detect changes in tracked attributes
src_hashed = source_df.withColumn("Src_hash_val", sha2(concat_ws("||", "customer_name", "city"), 256))
dim_hashed = dim_df.withColumn("Dim_hash_val", sha2(concat_ws("||", "customer_name", "city"), 256))



In [0]:
src_hashed.select("customer_id", "customer_name", "city", "Src_hash_val").display()
dim_hashed.select("customer_id", "customer_name", "city", "Dim_hash_val").display()

In [0]:

# Join on business key to detect changes
joined_df = src_hashed.join(dim_hashed, ["customer_id"], "left")
joined_df.display() 


In [0]:
from pyspark.sql.functions import col

records_to_expire = joined_df.filter(col("dim_hash_val").isNotNull()
    & (col("src_hash_val") != col("dim_hash_val"))
)

In [0]:
display(records_to_expire)

In [0]:
updates_expire = dim_df.join(records_to_expire, "customer_id") \
    .filter("is_current = 'Y'") \
    .withColumn("end_date", current_date()) \
    .withColumn("is_current", lit("N"))
updates_expire.display()

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

src = source_df.withColumnRenamed("hash_val", "src_hash_val").alias("src")
dim = dim_hashed.withColumnRenamed("hash_val", "dim_hash_val").alias("dim")

new_records = src.join(dim, "customer_id", "left") \
    .filter(
        col("dim.customer_id").isNull()
        | (col("src.src_hash_val") != col("dim.dim_hash_val"))
    ) \
    .drop("dim_hash_val") \
    .withColumn("customer_key", monotonically_increasing_id()) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit("Y"))


In [0]:
new_records.show(truncate=False)

In [0]:
# Keep unchanged old records
unchanged_records = dim_df.filter("is_current = 'Y'") \
    .join(records_to_expire, "customer_id", "left_anti")

# Expired + Unchanged + New = Final Dimension
final_dim = unchanged_records.unionByName(updates_expire).unionByName(new_records)


In [0]:
from pyspark.sql.functions import col

# Example: Remove table/alias prefix from column references
final_dim = final_dim.select(
    col("customer_id"),
    col("Dim_hash_val"),
    col("customer_name"),
    col("is_current"),
    col("start_date")
)

final_dim.write.mode("overwrite").format("delta").saveAsTable("DimCustomer")

In [0]:
final_dim.write.mode("overwrite").format("delta").saveAsTable("DimCustomer")
